# Döntési fa regresszor — Seoul Bike Trip Duration

**Felelős:** 3. hallgató

Ez a notebook egy DecisionTreeRegressor-t tanít be és hangol GridSearchCV-vel, majd menti az eredményt a közös `metrics.csv`-be.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Slityak/gepitan-seoul-bike-trip/blob/main/notebooks/03_decision_tree.ipynb)

## 1. Setup (Colab + lokális kompatibilis)

In [ ]:
import os
import sys

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')

    if not os.path.exists('/content/gepitan-beadando'):
        !git clone https://github.com/Slityak/gepitan-seoul-bike-trip.git /content/gepitan-beadando

    os.chdir('/content/gepitan-beadando')
    !git pull
    !pip install -q -r requirements.txt
else:
    if os.path.basename(os.getcwd()) == 'notebooks':
        os.chdir('..')

sys.path.insert(0, '.')
print(f'Working dir: {os.getcwd()}')
print(f'In Colab: {IN_COLAB}')

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.tree import DecisionTreeRegressor
from sklearn.model_selection import GridSearchCV
import joblib

from src.config import RANDOM_SEED, CV_FOLDS, MODELS_DIR, FIGURES_DIR, ensure_dirs
from src.data_io import load_v1_data
from src.evaluation import append_metrics, evaluate_model
from src.visualization import (
    plot_feature_importance,
    plot_predicted_vs_actual,
    plot_residuals,
)

ensure_dirs()

## 2. Adatbetöltés

A v1 final pipeline kimenetét töltjük be `load_v1_data()`-vel.

In [ ]:
X_train, X_test, y_train, y_test, feature_names = load_v1_data()

print(f'Train shape: {X_train.shape}, Test shape: {X_test.shape}')
print(f'Target — train: mean={y_train.mean():.2f}, std={y_train.std():.2f}, '
      f'min={y_train.min()}, max={y_train.max()}')

## 3. Baseline: alap döntési fa

Default `DecisionTreeRegressor` paraméterhangolás nélkül — referenciapont a tuned változathoz.

In [ ]:
dt_default = DecisionTreeRegressor(random_state=RANDOM_SEED)

result_default, y_pred_default = evaluate_model(
    dt_default,
    X_train, y_train, X_test, y_test,
    model_name='Decision Tree (default)',
    notes='no tuning',
)

print(f'MAE: {result_default.mae:.3f}')
print(f'RMSE: {result_default.rmse:.3f}')
print(f'R²: {result_default.r2:.3f}')
print(f'Train idő: {result_default.train_time_sec:.2f} s')

append_metrics(result_default)

## 4. GridSearchCV hangolás (sample-en) + refit a teljes train-en

A teljes 9.6M soros adaton közvetlenül futtatott GridSearchCV túl lassú, ezért:

1. `TUNE_SAMPLE_SIZE` random sample a train-ből → ezen futtatjuk a GridSearchCV-t a hiperparaméterek kiválasztására
2. A legjobb paraméterekkel egy újratanítás a teljes train-en

A grid is szűkebb mint az induló: a `max_depth=10` környéke a sweet spot, a `max_depth=None` lassú nagyon nagy adaton és úgyis túltanul.

In [ ]:
TUNE_SAMPLE_SIZE = 500_000  # train-en belüli random sample a tuninghoz

param_grid = {
    'max_depth': [8, 10, 12, 15],
    'min_samples_leaf': [10, 50, 100],
    'min_samples_split': [2],
}

# random sample a train-ből
n_tune = min(TUNE_SAMPLE_SIZE, len(X_train))
rng = np.random.default_rng(RANDOM_SEED)
tune_idx = rng.choice(len(X_train), size=n_tune, replace=False)
X_tune, y_tune = X_train[tune_idx], y_train[tune_idx]

grid = GridSearchCV(
    estimator=DecisionTreeRegressor(random_state=RANDOM_SEED),
    param_grid=param_grid,
    cv=CV_FOLDS,
    scoring='neg_mean_absolute_error',
    n_jobs=-1,
    verbose=1,
)
grid.fit(X_tune, y_tune)

print(f'\nTune sample size: {n_tune}')
print(f'Legjobb paraméterek: {grid.best_params_}')
print(f'CV legjobb score (neg MAE): {grid.best_score_:.3f}')

# refit a teljes train-en a legjobb paraméterekkel
best_model = DecisionTreeRegressor(**grid.best_params_, random_state=RANDOM_SEED)
result_tuned, y_pred_tuned = evaluate_model(
    best_model,
    X_train, y_train, X_test, y_test,
    model_name='Decision Tree (tuned)',
    notes=f'GridSearchCV cv={CV_FOLDS} on n={n_tune}, refit on full train',
)

print(f'\nTeszt MAE: {result_tuned.mae:.3f}')
print(f'Teszt RMSE: {result_tuned.rmse:.3f}')
print(f'Teszt R²: {result_tuned.r2:.3f}')

append_metrics(result_tuned)

MODELS_DIR.mkdir(parents=True, exist_ok=True)
joblib.dump(best_model, MODELS_DIR / 'decision_tree_tuned.joblib')

## 5. Vizualizációk

In [ ]:
plot_predicted_vs_actual(
    y_test, y_pred_tuned,
    model_name='Decision Tree (tuned)',
    save_path='dt_predicted_vs_actual.png',
)
plt.show()

In [ ]:
plot_residuals(
    y_test, y_pred_tuned,
    model_name='Decision Tree (tuned)',
    save_path='dt_residuals.png',
)
plt.show()

In [ ]:
plot_feature_importance(
    feature_names=feature_names,
    importances=best_model.feature_importances_,
    model_name='Decision Tree (tuned)',
    top_n=20,
    save_path='dt_feature_importance.png',
)
plt.show()

## 6. Tanulságok

_Hát az van_